# Prepare Derm1M dataset for VLM training

Convert labeled Derm1M pretrain rows into canonical JSONL.

Kept as an **experimental / exploration** resource. It is **not** the primary dataset of the main study (PAD-UFES-20).

- **Source:** `data/datasets/Derm1M/` — `Derm1M_v2_pretrain.csv` + `images/<source>/...`
- **Output:** `data/processed/derm1m/clinical_context/`
- **Official validation CSV has no disease labels** — held out; splits are carved from labeled pretrain (80/10/10, grouped by `filename`, not stratified — too many singleton classes)
- **Dropped:** `disease_label == "no definitive diagnosis"`; duplicate `filename`s keep the first row
- **Ignored in the user prompt:** `caption` / `truncated_caption` (often leak the diagnosis)
- **Clinical fields:** age, gender, body_location, symptoms, skin_concept (placeholders treated as missing)
- Set `LIMIT` for a dry run


## 1. Setup


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise FileNotFoundError(f"Could not find repo root (src/) from {Path.cwd()}")

sys.path.insert(0, str(ROOT / "src"))

from vlm_ft.data.canonical import derm1m_image_rel, derm1m_prompt_and_label
from vlm_ft.data.prepare import (
    count_existing_images,
    load_derm1m_labeled,
    preview_processed,
    print_source_eda,
    rel_to_root,
    require_source,
    rows_to_samples,
    split_by_group,
    write_processed_dataset,
)

CURRENT_SOURCES = "PAD, ISIC18, HC, Derm1M, MILK10K"

DERM_ROOT = ROOT / "data/datasets/Derm1M"
OUT_DIR = ROOT / "data/processed/derm1m/clinical_context"
LIMIT: int | None = None  # e.g. 1000 for a dry run
require_source(DERM_ROOT, expected=CURRENT_SOURCES)


## 2. Load


In [ ]:
df = load_derm1m_labeled(DERM_ROOT, limit=LIMIT)
print("labeled unique images:", len(df))
print("official validation rows (unlabeled):", sum(1 for _ in open(DERM_ROOT / "Derm1M_v2_validation.csv", encoding="utf-8")) - 1)
df.head()


## 3. EDA


In [ ]:
ok, total = count_existing_images(df, "image_rel", DERM_ROOT, limit=200)
print(f"image existence (first 200): {ok}/{total}")
print("sources:\n", df["source"].value_counts().head(10).to_string() if "source" in df.columns else "n/a")

splits = split_by_group(df, group_col="filename", label_col=None)
eda = pd.concat(splits.values(), ignore_index=True)
print_source_eda(
    eda,
    label_col="disease_label",
    metadata_cols=["age", "gender", "body_location", "symptoms", "skin_concept"],
)


## 4. Convert to canonical JSONL


In [ ]:
DERM_ROOT_REL = rel_to_root(DERM_ROOT, ROOT)
samples = {
    split: rows_to_samples(
        frame,
        DERM_ROOT,
        prompt_and_label=derm1m_prompt_and_label,
        image_rel=derm1m_image_rel,
    )
    for split, frame in splits.items()
}
write_processed_dataset(
    name="derm1m/clinical_context",
    out_dir=OUT_DIR,
    split_samples=samples,
    image_root_rel=DERM_ROOT_REL,
)


## 5. Validate and preview


In [ ]:
preview_processed(OUT_DIR, "derm1m/clinical_context")
